# 01a — Load LEDGAR to Delta

Ingests the LEDGAR subset of LexGLUE (~80,000 SEC contract provisions labeled across
100 legal categories) from Hugging Face and lands it as the project's foundation table.

**What this notebook does**
1. Downloads the `lighteval/lexglue` LEDGAR dataset with canonical train / validation / test splits (60k / 10k / 10k)
2. Cleans and standardizes columns (`provision_text`, `category_label`, `split`, `provision_id`) and writes the Delta table **`ledgar_lexglue`**
3. Runs data-quality validation: row counts, split integrity, label distribution, null/empty/duplicate checks

For rapid eval iteration, subsample the test split in `03_evaluation` — do not mutate this table.


In [0]:
# Configuration widgets (values re-read after Python restart)
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")

In [0]:
%pip install datasets==3.2.0 huggingface_hub==0.26.5

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_NAME = f"{CATALOG}.{SCHEMA}.ledgar_lexglue"

print(f"Target table: {TABLE_NAME}")

Target table: workspace.default.ledgar_lexglue


In [0]:
from datasets import load_dataset

dataset = load_dataset("lighteval/lexglue", "ledgar")
dataset

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:56: UserWarning: The cache_dir for this dataset is /tmp/.hf.data.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/Volumes/']
  warnings.warn(warning_message)


README.md: 0.00B [00:00, ?B/s]

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:24: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


train-00000-of-00001.parquet:   0%|          | 0.00/22.6M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/3.72M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 60000
    })
    validation: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
})

In [0]:
EXPECTED_SPLITS = {"train": 60_000, "validation": 10_000, "test": 10_000}
actual_splits = {split: len(dataset[split]) for split in EXPECTED_SPLITS}

if actual_splits != EXPECTED_SPLITS:
    print(f"Split mismatch — expected {EXPECTED_SPLITS}, got {actual_splits}")
else:
    print("Split counts verified:", actual_splits)

Split counts verified: {'train': 60000, 'validation': 10000, 'test': 10000}


In [0]:
print(dataset)
print(dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 60000
    })
    validation: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
})
['input', 'references', 'gold']


In [0]:
from pyspark.sql import functions as F

spark_dfs = []

for split_name in dataset.keys():
    pdf = dataset[split_name].to_pandas()
    sdf = spark.createDataFrame(pdf)
    sdf = sdf.withColumn("split", F.lit(split_name))
    spark_dfs.append(sdf)

ledgar_df = spark_dfs[0]

for sdf in spark_dfs[1:]:
    ledgar_df = ledgar_df.unionByName(sdf)

display(ledgar_df.limit(10))
ledgar_df.printSchema()

input,references,gold,split
"Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.","List(Qualifications, Modifications, Titles, Authority, Effective Dates, Counterparts, Agreements, Releases, Brokers, No Defaults, Severability, Authorizations, Integration, Terms, Insurances, Transactions With Affiliates, Indemnifications, Expenses, Organizations, Disability, Jurisdictions, Records, Further Assurances, Duties, Submission To Jurisdiction, Payments, Vacations, Assigns, Enforcements, Sanctions, Waivers)",List(Waivers),train
"No ERISA Event has occurred or is reasonably expected to occur that, when taken together with all other such ERISA Events for which liability is reasonably expected to occur, could reasonably be expected to result in a Material Adverse Effect. Neither Borrower nor any ERISA Affiliate maintains or contributes to or has any obligation to maintain or contribute to any Multiemployer Plan or Plan, nor otherwise has any liability under Title IV of ERISA.","List(Interests, Enforcements, No Conflicts, Consents, Approvals, Applicable Laws, Publicity, Venues, Binding Effects, Costs, Payments, Participations, Liens, Disability, Intellectual Property, Sales, Amendments, Counterparts, Agreements, Headings, No Waivers, Existence, Anti-Corruption Laws, General, Brokers, Tax Withholdings, Enforceability, Financial Statements, Waivers, Cooperation, Erisa)",List(Erisa),train
"This Amendment may be executed by one or more of the parties hereto on any number of separate counterparts, and all of said counterparts taken together shall be deemed to constitute one and the same instrument. This Amendment may be delivered by facsimile or other electronic transmission of the relevant signature pages hereof.","List(Warranties, Releases, Interests, Subsidiaries, Enforcements, Qualifications, Entire Agreements, Authorizations, Effective Dates, Closings, Compliance With Laws, Expenses, Construction, Notices, General, Binding Effects, Approvals, Payments, Positions, Severability, Cooperation, Confidentiality, Agreements, Venues, Non-Disparagement, Jurisdictions, No Defaults, Survival, Effectiveness, Sales, Counterparts)",List(Counterparts),train
"From time to time, as and when required by the Surviving Corporation or by its successors or assigns, there shall be executed and delivered on behalf of Ashford (DE) such deeds and other instruments, and there shall be taken or caused to be taken by it all such further and other action, as shall be appropriate, advisable or necessary in order to vest, perfect or confirm, of record or otherwise, in the Surviving Corporation the title to and possession of all property, interests, assets, rights, privileges, immunities, powers, franchises and authority of Ashford (DE), and otherwise to carry out the purposes of this Agreement. The officers and directors of the Surviving Corporation are fully authorized in the name and on behalf of Ashford (DE) or otherwise, to take any and all such action and to execute and deliver any and all such deeds and other instruments.","List(Warranties, Books, Qualifications, Publicity, Non-Disparagement, Amendments, Forfeitures, Base Salary, Benefits, Miscellaneous, Agreements, Financial Statements, Representations, Subsidiaries, Entire Agreements, Interpretations, Positions, Employment, Headings, Authority, Tax Withholdings, Waiver Of Jury Trials, Change In Control, Waivers, No Conflicts, General, Litigations, Indemnity, Anti-Corruption Laws, Consent To Jurisdiction, Further Assurances)",List(Further Assurances),train
"Commencing March 7, 2016 and during the Employment Period, the Company shall pay to the Executive a base salary at the rate of no less than $750,000 per calendar year (the “Base 

root
 |-- input: string (nullable = true)
 |-- references: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- gold: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- split: string (nullable = false)



In [0]:
ledgar_clean_df = (
    ledgar_df
    .select(
        F.col("input").alias("provision_text"),
        F.col("gold").alias("category_label"),
        F.col("split")
    )
    .withColumn("provision_id", F.monotonically_increasing_id())
    .withColumn("text_length", F.length("provision_text"))
    .withColumn("ingested_at", F.current_timestamp())
)

display(ledgar_clean_df.limit(10))

provision_text,category_label,split,provision_id,text_length,ingested_at
"Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.",List(Waivers),train,0,338,2026-06-19T02:07:56.191Z
"No ERISA Event has occurred or is reasonably expected to occur that, when taken together with all other such ERISA Events for which liability is reasonably expected to occur, could reasonably be expected to result in a Material Adverse Effect. Neither Borrower nor any ERISA Affiliate maintains or contributes to or has any obligation to maintain or contribute to any Multiemployer Plan or Plan, nor otherwise has any liability under Title IV of ERISA.",List(Erisa),train,1,452,2026-06-19T02:07:56.191Z
"This Amendment may be executed by one or more of the parties hereto on any number of separate counterparts, and all of said counterparts taken together shall be deemed to constitute one and the same instrument. This Amendment may be delivered by facsimile or other electronic transmission of the relevant signature pages hereof.",List(Counterparts),train,2,328,2026-06-19T02:07:56.191Z
"From time to time, as and when required by the Surviving Corporation or by its successors or assigns, there shall be executed and delivered on behalf of Ashford (DE) such deeds and other instruments, and there shall be taken or caused to be taken by it all such further and other action, as shall be appropriate, advisable or necessary in order to vest, perfect or confirm, of record or otherwise, in the Surviving Corporation the title to and possession of all property, interests, assets, rights, privileges, immunities, powers, franchises and authority of Ashford (DE), and otherwise to carry out the purposes of this Agreement. The officers and directors of the Surviving Corporation are fully authorized in the name and on behalf of Ashford (DE) or otherwise, to take any and all such action and to execute and deliver any and all such deeds and other instruments.",List(Further Assurances),train,3,870,2026-06-19T02:07:56.191Z
"Commencing March 7, 2016 and during the Employment Period, the Company shall pay to the Executive a base salary at the rate of no less than $750,000 per calendar year (the “Base Salary”), less applicable deductions, and prorated for any partial month or year, as applicable. The Base Salary shall be reviewed for increase by the Compensation Committees of AFG and AAC (the “Compensation Committees”) no less frequently than annually and may be increased in the discretion of the Compensation Committees. Any such adjusted Base Salary shall constitute the “Base Salary” for purposes of this Agreement. The Base Salary shall be paid in substantially equal installments in accordance with AAC’s regular payroll procedures. The Executive’s Base Salary may not be decreased during the Employment Period. The Company shall provide the Executive with a payment in an amount equal to the difference between (i) the Base Salary payments the Executive would have received had he been paid at the rate set forth in this Section 4(a) during the period commencing on March 7, 2016 and ending on the Effective Date hereof and (ii) the actual salary payments made to the Executive during such period, payable in a lump sum on a regular payroll date as soon as practicable following the Effective Date.",List(Base Salary),train,4,1286,2026-06-19T02:07:56.191Z
"All notices required or permitted under this Agreement will be in writing, will reference this Agreement, and will be deemed given: (i) when delivered personally; (ii) one (1) business day after deposit with a nationally-recognized express courier, with written confirmation of receipt; or (iii) three (3) business days after having been sent by registered or cert

In [0]:
(
    ledgar_clean_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Wrote {TABLE_NAME}")

Wrote workspace.default.ledgar_lexglue


In [0]:
df = spark.table(TABLE_NAME)

display(df.limit(10))
df.printSchema()

provision_text,category_label,split,provision_id,text_length,ingested_at
"For all services rendered by Sorensen and all covenants and conditions undertaken by her pursuant to this Agreement, the Company shall pay Sorensen in accordance with its normal payroll practices (but not less frequently than monthly) a base salary equal to $5,000 per month, less applicable withholdings (the “ Base Salary ”). Such Base Salary shall be reviewed from time-to-time but not less than annually by the Board or the Compensation Committee of the Board, which shall make recommendations to adjust the Base Salary, if necessary, based upon appropriate applicable performance metrics.",List(Base Salary),train,10000,594,2026-06-19T02:08:05.272Z
"Your current, annualized base salary is set at Three Hundred Thirty Five Thousand Dollars and Zero Cents ($335,000.00), less all applicable taxes and withholdings, payable in installments in accordance with the Company’s regular payroll practices.",List(Base Salary),train,10001,247,2026-06-19T02:08:05.272Z
"This Agreement may be terminated by any Purchaser, as to such Purchaser’s obligations hereunder only and without any effect whatsoever on the obligations between the Company and the other Purchasers, by written notice to the other parties, if the Closing has not been consummated on or before August __, 2016; provided , however , that such termination will not affect the right of any party to sue for any breach by any other party (or parties).",List(Terminations),train,10002,446,2026-06-19T02:08:05.272Z
"No failure or delay on the part of the Administrative Agent or any Lender in exercising any right, power or privilege hereunder or under any other Loan Document and no course of dealing between the Borrower and the Administrative Agent or any Lender shall operate as a waiver thereof; nor shall any single or partial exercise of any right, power or privilege hereunder or under any other Loan Document preclude any other or further exercise thereof or the exercise of any other right, power or privilege hereunder or thereunder. No notice to or demand on the Borrower in any case shall entitle the Borrower to any other or further notice or demand in similar or other circumstances or constitute a waiver of the rights of the Administrative Agent or the Lenders to any other or further action in any circumstances without notice or demand. Without limiting the generality of the foregoing, the making of a Loan shall not be construed as a waiver of any Default or Event of Default, regardless of whether the Administrative Agent or any Lender may have had notice or knowledge of such Default or Event of Default at the time. The rights and remedies herein expressly provided are cumulative and not exclusive of any rights or remedies that the Administrative Agent or any Lender would otherwise have.",List(No Waivers),train,10003,1299,2026-06-19T02:08:05.272Z
"Interest shall accrue on ESH REIT reimbursement obligation under Section 1 at the relevant applicable federal rate as determined under Section 1274(d) of the Internal Revenue Code of 1985, as amended.",List(Interests),train,10004,200,2026-06-19T02:08:05.272Z
"Employee’s title shall be Chief Executive Officer of Altair Nanotechnologies, Inc. Employee’s duties shall include such duties as are specifically assigned or delegated to Employee by the Board of Directors of any Consolidated Company (any such Board of Directors, the “Board”) and such other duties as are typically performed by an employee with the same position as Employee. Employee acknowledges that, subject to Section 6.3(c), the Board may change, increase or decrease Employee’s title, position and/or duties from time to time its discretion and may appoint Employee as employee of another Consolidated Company, which employment is governed by this Agreement. Employee shall diligently execute his duties and shall devote his full time, skills and efforts to such duties during ordinary working hours. Employee shall faithfu

root
 |-- provision_text: string (nullable = true)
 |-- category_label: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- split: string (nullable = true)
 |-- provision_id: long (nullable = true)
 |-- text_length: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
print("Total rows:", df.count())

Total rows: 80000


In [0]:
display(
    df.groupBy("split")
      .count()
      .orderBy("split")
)

split,count
test,10000
train,60000
validation,10000


In [0]:
display(
    df.groupBy("category_label")
      .count()
      .orderBy(F.desc("count"))
)

category_label,count
List(Governing Laws),4243
List(Counterparts),3346
List(Notices),3313
List(Entire Agreements),3105
List(Severability),2552
List(Survival),1951
List(Amendments),1948
List(Assignments),1730
List(Expenses),1577
List(Terms),1511


In [0]:
display(
    df.agg(F.countDistinct("category_label").alias("distinct_category_labels"))
)

distinct_category_labels
100


In [0]:
display(
    df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in ["provision_text", "category_label", "split"]
    ])
)

provision_text,category_label,split
0,0,0


In [0]:
display(
    df.filter(F.trim(F.col("provision_text")) == "")
)

provision_text,category_label,split,provision_id,text_length,ingested_at


In [0]:
display(
    df.groupBy("provision_text")
      .count()
      .filter(F.col("count") > 1)
      .orderBy(F.desc("count"))
)

provision_text,count


## Data Quality Summary

The LEDGAR subset of LexGLUE was successfully ingested from Hugging Face and stored as a Delta table within Databricks.

### Validation Results
- Canonical LexGLUE splits preserved: train (60,000), validation (10,000), test (10,000)
- Label distribution analyzed across all categories
- Distinct category count, null, empty-text, and duplicate checks completed

### Data Quality Observations
- The dataset is well-structured and requires minimal preprocessing prior to downstream NLP workflows.
- Legal provision text and category labels are consistently populated across records.
- Category frequencies are imbalanced, which is expected in legal-domain classification datasets.
- The dataset consists of publicly available SEC EDGAR filings and does not contain confidential client intake information.
- Suitable for legal text classification, RAG, and agent evaluation within this project.